# Sanjavan Ghodasara

**Cluster 2 Stacking Model**

**Group 2** | Cluster 2: 706 companies, 133 bankrupt (18.84% rate)

Note: Cluster 2 has the **highest bankruptcy rate of all 6 clusters**, which makes the
20% sparsity cap tight — only ~1.16% room for false positives above the base rate.
After extensive experimentation, the winning architecture pairs boosted/bagged trees
with a **distance-weighted KNN** that contributes local density signal, combined
through a mildly-regularized logistic regression meta-learner.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.feature_selection import mutual_info_classif

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CLUSTER_DIR = os.path.join(DATA_DIR, 'clusters')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
SUBGROUP_MODEL_DIR = os.path.join(MODEL_DIR, 'subgroup_models')
os.makedirs(SUBGROUP_MODEL_DIR, exist_ok=True)

In [2]:
def custom_accuracy(y_true, y_pred):
    """Project metric: acc = TT / (TT + TF) = recall of bankrupt class.
    TT = correctly predicted bankrupt, TF = bankrupt predicted as non-bankrupt."""
    tt = ((y_true == 1) & (y_pred == 1)).sum()
    tf = ((y_true == 1) & (y_pred == 0)).sum()
    if tt + tf == 0:
        return 0.0
    return tt / (tt + tf)

def sparsity_check(y_pred, threshold=0.20):
    """Check if < 20% of predictions are bankrupt."""
    rate = y_pred.mean()
    return rate < threshold, rate

def evaluate_model(y_true, y_pred, label=''):
    """Print evaluation metrics."""
    acc = custom_accuracy(y_true, y_pred)
    ok, rate = sparsity_check(y_pred)
    cm = confusion_matrix(y_true, y_pred)
    tt = int(((y_true == 1) & (y_pred == 1)).sum())
    tf = int(((y_true == 1) & (y_pred == 0)).sum())
    print(f'--- {label} ---')
    print(f'TT (bankrupt predicted bankrupt):     {tt}')
    print(f'TF (bankrupt predicted non-bankrupt): {tf}')
    print(f'TT + TF = {tt + tf} (total bankrupt)')
    print(f'Custom Accuracy (TT/(TT+TF)): {acc:.4f}')
    print(f'Sparsity: {rate:.4f} ({"PASS" if ok else "FAIL"})')
    print(f'Confusion Matrix:\n{cm}')
    print(f'Classification Report:\n{classification_report(y_true, y_pred, zero_division=0)}')
    return acc, rate

---
## Load & Inspect Data

In [3]:
# Load cluster 2 data
df2 = pd.read_csv(os.path.join(CLUSTER_DIR, 'cluster_2.csv'))
y2 = df2['Bankrupt?'].values
X2 = df2.drop(columns=['Bankrupt?'])
feature_names = X2.columns.tolist()

print(f'Cluster 2: {X2.shape[0]} samples, {X2.shape[1]} features')
print(f'Bankrupt: {int(y2.sum())} ({y2.mean():.4f})')
print(f'Non-bankrupt: {int((y2 == 0).sum())}')
print(f'\nBankruptcy rate: {y2.mean()*100:.2f}% — leaves ~1.16% room for FPs under the 20% sparsity cap')

Cluster 2: 706 samples, 95 features
Bankrupt: 133 (0.1884)
Non-bankrupt: 573

Bankruptcy rate: 18.84% — leaves ~1.16% room for FPs under the 20% sparsity cap


## Experimentation Summary

Before selecting the final architecture, I ran **101 configurations** across five axes:

| Axis | What I varied |
|---|---|
| **Feature engineering** | Raw MI, MI + correlation de-dup, engineered ratios/products/differences, log-transforms |
| **Base models** | RF+GB+SVM (A), RF+GB+LR (B), RF+XGB+SVM (C), **RF+GB+KNN (D)**, RF+GB+SVM+ET (E) |
| **Class imbalance** | No SMOTE, SMOTE(0.25), SMOTE(0.30), SMOTE(0.50) inside CV folds only |
| **Meta learners** | LR (various C), Ridge, linear-SVM, GradientBoosting |
| **Threshold** | Sweep 0.05–0.95, step 0.01, requiring sparsity<20% on BOTH CV and train |

### Top 10 configs (all pass sparsity on both CV and training)

| Rank | Config | N | Thresh | CV TT/TF | CV Acc | Train TT/TF | Train Acc | Score |
|---|---|---|---|---|---|---|---|---|
| **1** | **D: RF+GB+KNN(k=20) + LR(C=2.0)** | **6** | **0.32** | **67/66** | **0.5038** | **132/1** | **0.9925** | **0.7632** |
| 2 | D: RF+GB+KNN(k=20) + LR(C=0.5) | 6 | 0.30 | 67/66 | 0.5038 | 131/2 | 0.9850 | 0.7610 |
| 3 | D: RF+GB+KNN(k=20) + LR(bal,C=0.1) | 6 | 0.60 | 67/66 | 0.5038 | 131/2 | 0.9850 | 0.7610 |
| 4 | D: RF+GB+KNN(k=20) + LR(C=0.1) | 6 | 0.26 | 67/66 | 0.5038 | 130/3 | 0.9774 | 0.7587 |
| 5 | D: RF+GB+KNN(k=15) + LR(C=1.0) | 6 | 0.31 | 67/66 | 0.5038 | 129/4 | 0.9699 | 0.7565 |
| 6 | D: RF+GB+KNN(k=25) | 6 | 0.26 | 65/68 | 0.4887 | 131/2 | 0.9850 | 0.7550 |
| 7 | D: RF+GB+KNN(k=15) + SVM-linear | 6 | 0.33 | 66/67 | 0.4962 | 131/2 | 0.9850 | 0.7580 |
| 8 | D: RF+GB+KNN(k=30) | 6 | 0.26 | 65/68 | 0.4887 | 130/3 | 0.9774 | 0.7527 |
| 9 | D: RF+GB+KNN(k=15), SMOTE(0.3) | 6 | 0.32 | 63/70 | 0.4737 | 132/1 | 0.9925 | 0.7512 |
| 10 | A: RF+GB+SVM, SMOTE(0.5) | 6 | 0.46 | 67/66 | 0.5038 | 96/37 | 0.7218 | 0.6820 |
| baseline | A: RF+GB+SVM (original) | 6 | 0.27 | 55/78 | 0.4135 | 93/40 | 0.6992 | 0.6369 |

### Key findings

1. **Swapping SVM → KNN in the base set was the single biggest win** (CV acc jumps from 0.41 → 0.50). KNN contributes local-density information that trees and SVM both miss.
2. **Raw MI + correlation de-duplication beats engineered features.** Products/ratios of top MI features did not help because the top features are already leverage ratios; more leverage features added noise.
3. **Light SMOTE is roughly neutral** with KNN in the stack. The k-NN already functions like an implicit density model.
4. **Meta-learner regularization matters: LR(C=2.0) edges out C=0.1 by ~0.005 rank score** (less penalty on the KNN signal).
5. **N=6 hits the sweet spot.** N=5 drops CV acc noticeably, N=8 hits the same CV acc but loses 0.02 on feature score.

## Feature Selection

Rank all 95 features by mutual information, then greedily select while dropping any
feature whose pairwise |correlation| with an already-selected feature exceeds 0.85.
This is critical for cluster 2 — the top 5 raw-MI features are all leverage/debt ratios
that measure essentially the same signal.

In [4]:
# Rank all features by mutual information
mi_scores = mutual_info_classif(X2.values, y2, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=feature_names).sort_values(ascending=False)

print('Top 15 features by Mutual Information:')
for i, (feat, score) in enumerate(mi_series.head(15).items()):
    print(f'  {i+1:2d}. {feat[:55]:55s}  MI = {score:.4f}')

# Greedy diverse selection
corr_matrix = X2.corr().abs()
CORR_THRESH = 0.85
selected = []
for feat in mi_series.index:
    if not selected or not any(corr_matrix.loc[feat, s] > CORR_THRESH for s in selected):
        selected.append(feat)
    if len(selected) >= 10:
        break

N_FEATURES = 6
top_features = selected[:N_FEATURES]
X2_sel = df2[top_features].values

print(f'\nSelected {N_FEATURES} diverse features (pairwise |r| < {CORR_THRESH}):')
for i, feat in enumerate(top_features):
    print(f'  {i+1}. {feat[:55]:55s}  MI = {mi_series[feat]:.4f}')

Top 15 features by Mutual Information:
   1. Total debt/Total net worth                               MI = 0.1025
   2. Equity to Liability                                      MI = 0.1016
   3. Debt ratio %                                             MI = 0.1014
   4. Net worth/Assets                                         MI = 0.1013
   5. Liability to Equity                                      MI = 0.0984
   6. Borrowing dependency                                     MI = 0.0799
   7. Net profit before tax/Paid-in capital                    MI = 0.0709
   8. Persistent EPS in the Last Four Seasons                  MI = 0.0629
   9. Net Income to Stockholder's Equity                       MI = 0.0547
  10. Working Capital to Total Assets                          MI = 0.0533
  11. Net Value Per Share (B)                                  MI = 0.0513
  12. Working Capital/Equity                                   MI = 0.0495
  13. Current Ratio                                          

## Stacking Model Definition

**Base models (3 — with model-type diversity):**
1. **Random Forest** — bagged trees, `class_weight='balanced'`
2. **Gradient Boosting** — boosted trees, handles imbalance through the loss
3. **KNN (k=20, distance-weighted)** — non-parametric local density; contributes unique signal trees cannot capture

**Meta model:** `LogisticRegression(C=2.0)` — mild regularization, unweighted

**Preprocessing:** `StandardScaler` (per CV fold to avoid leakage) — KNN requires scaled features.

In [5]:
BEST_THRESH = 0.32

print(f'Using {N_FEATURES} features, threshold={BEST_THRESH:.2f}')

base_estimators = [
    ('rf', RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05,
        min_samples_leaf=10, random_state=RANDOM_STATE)),
    ('knn', KNeighborsClassifier(
        n_neighbors=20, weights='distance', n_jobs=-1)),
]

meta_learner = LogisticRegression(
    C=2.0, max_iter=1000, random_state=RANDOM_STATE)

print('Stacking classifier: RF + GB + KNN base models + LR(C=2.0) meta-learner.')

Using 6 features, threshold=0.32
Stacking classifier: RF + GB + KNN base models + LR(C=2.0) meta-learner.


## Individual Base-Model Diagnostics

Before stacking, print confusion matrices for each base model on its own
(5-fold stratified `cross_val_predict` with per-fold StandardScaler). This confirms
each base learner is contributing distinct signal — not all three are making the same
mistakes. These are diagnostic (standard 0.5 threshold) and don't affect the final
stacked predictions.

In [6]:
# Per-base-model CV predictions using cross_val_predict with per-fold scaling.
# We reuse the outer 5-fold CV splitter for a fair comparison against the stack.
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def base_cv_predict(model_factory, X_raw, y):
    """5-fold CV predictions with per-fold StandardScaler (no leakage)."""
    preds = np.zeros(len(y), dtype=int)
    for train_idx, val_idx in outer_cv.split(X_raw, y):
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_raw[train_idx])
        X_val = sc.transform(X_raw[val_idx])
        m = model_factory()
        m.fit(X_tr, y[train_idx])
        preds[val_idx] = m.predict(X_val)
    return preds

# RF
rf_preds_cv = base_cv_predict(
    lambda: RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    X2_sel, y2)
evaluate_model(y2, rf_preds_cv, 'Base: Random Forest (CV preds, thresh=0.5)')

# GB
gb_preds_cv = base_cv_predict(
    lambda: GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05,
        min_samples_leaf=10, random_state=RANDOM_STATE),
    X2_sel, y2)
evaluate_model(y2, gb_preds_cv, 'Base: Gradient Boosting (CV preds, thresh=0.5)')

# KNN
knn_preds_cv = base_cv_predict(
    lambda: KNeighborsClassifier(n_neighbors=20, weights='distance', n_jobs=-1),
    X2_sel, y2)
evaluate_model(y2, knn_preds_cv, 'Base: KNN (k=20, distance, CV preds, thresh=0.5)')

--- Base: Random Forest (CV preds, thresh=0.5) ---
TT (bankrupt predicted bankrupt):     78
TF (bankrupt predicted non-bankrupt): 55
TT + TF = 133 (total bankrupt)
Custom Accuracy (TT/(TT+TF)): 0.5865
Sparsity: 0.2847 (FAIL)
Confusion Matrix:
[[450 123]
 [ 55  78]]
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.79      0.83       573
           1       0.39      0.59      0.47       133

    accuracy                           0.75       706
   macro avg       0.64      0.69      0.65       706
weighted avg       0.80      0.75      0.77       706



--- Base: Gradient Boosting (CV preds, thresh=0.5) ---
TT (bankrupt predicted bankrupt):     41
TF (bankrupt predicted non-bankrupt): 92
TT + TF = 133 (total bankrupt)
Custom Accuracy (TT/(TT+TF)): 0.3083
Sparsity: 0.1062 (PASS)
Confusion Matrix:
[[539  34]
 [ 92  41]]
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.94      0.90       573
           1       0.55      0.31      0.39       133

    accuracy                           0.82       706
   macro avg       0.70      0.62      0.64       706
weighted avg       0.80      0.82      0.80       706

--- Base: KNN (k=20, distance, CV preds, thresh=0.5) ---
TT (bankrupt predicted bankrupt):     21
TF (bankrupt predicted non-bankrupt): 112
TT + TF = 133 (total bankrupt)
Custom Accuracy (TT/(TT+TF)): 0.1579
Sparsity: 0.0538 (PASS)
Confusion Matrix:
[[556  17]
 [112  21]]
Classification Report:
              precision    recall  f1-score   support

           0       0.83      

(np.float64(0.15789473684210525), np.float64(0.053824362606232294))

## Cross-Validation (Stacked Model)

5-fold stratified CV. StandardScaler is fit on each training fold and applied to
the corresponding validation fold — no scaling leakage.

In [7]:
# 5-fold stratified CV with per-fold StandardScaler + probability threshold tuning
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y2_cv_probas = np.zeros(len(y2))

for fold, (train_idx, val_idx) in enumerate(cv.split(X2_sel, y2)):
    sc_fold = StandardScaler()
    X_train_cv = sc_fold.fit_transform(X2_sel[train_idx])
    X_val_cv = sc_fold.transform(X2_sel[val_idx])
    y_train_cv, y_val_cv = y2[train_idx], y2[val_idx]

    stacker_fold = StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(
                n_estimators=200, max_depth=6, min_samples_leaf=5,
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
            ('gb', GradientBoostingClassifier(
                n_estimators=150, max_depth=3, learning_rate=0.05,
                min_samples_leaf=10, random_state=RANDOM_STATE)),
            ('knn', KNeighborsClassifier(
                n_neighbors=20, weights='distance', n_jobs=-1)),
        ],
        final_estimator=LogisticRegression(
            C=2.0, max_iter=1000, random_state=RANDOM_STATE),
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        stack_method='predict_proba',
        n_jobs=-1
    )
    stacker_fold.fit(X_train_cv, y_train_cv)
    y2_cv_probas[val_idx] = stacker_fold.predict_proba(X_val_cv)[:, 1]
    print(f'Fold {fold+1}: val positives={int(y_val_cv.sum())}, avg proba bankrupt={y2_cv_probas[val_idx].mean():.4f}')

# Threshold sweep — maximize CV custom accuracy s.t. sparsity < 20% on BOTH CV and train
print('\n--- Threshold Sweep (CV) ---')
best_thresh, best_acc = 0.5, 0.0
for thresh in np.arange(0.05, 0.96, 0.01):
    preds = (y2_cv_probas >= thresh).astype(int)
    acc = custom_accuracy(y2, preds)
    _, rate = sparsity_check(preds)
    if rate < 0.20 and acc >= best_acc:
        best_acc = acc
        best_thresh = thresh
    if round(thresh, 2) in [0.10, 0.15, 0.20, 0.25, 0.30, 0.32, 0.35, 0.40, 0.45, 0.50]:
        tt = int(((y2 == 1) & (preds == 1)).sum())
        tf = int(((y2 == 1) & (preds == 0)).sum())
        print(f'  thresh={thresh:.2f}: TT={tt:3d}, TF={tf:3d}, acc={acc:.4f}, sparsity={rate:.4f} {"PASS" if rate < 0.20 else "FAIL"}')

print(f'\nBest threshold: {BEST_THRESH:.2f} (chosen to also pass training sparsity)')
y2_cv_preds = (y2_cv_probas >= BEST_THRESH).astype(int)
evaluate_model(y2, y2_cv_preds, f'Stacked: Cluster 2 — 5-Fold CV (thresh={BEST_THRESH:.2f})')

Fold 1: val positives=27, avg proba bankrupt=0.1852


Fold 2: val positives=26, avg proba bankrupt=0.2015


Fold 3: val positives=26, avg proba bankrupt=0.1819


Fold 4: val positives=27, avg proba bankrupt=0.1894


Fold 5: val positives=27, avg proba bankrupt=0.2039

--- Threshold Sweep (CV) ---
  thresh=0.10: TT=120, TF= 13, acc=0.9023, sparsity=0.5850 FAIL
  thresh=0.15: TT=107, TF= 26, acc=0.8045, sparsity=0.4575 FAIL
  thresh=0.20: TT= 91, TF= 42, acc=0.6842, sparsity=0.3669 FAIL
  thresh=0.25: TT= 82, TF= 51, acc=0.6165, sparsity=0.2975 FAIL
  thresh=0.30: TT= 72, TF= 61, acc=0.5414, sparsity=0.2167 FAIL
  thresh=0.32: TT= 67, TF= 66, acc=0.5038, sparsity=0.1969 PASS
  thresh=0.35: TT= 62, TF= 71, acc=0.4662, sparsity=0.1700 PASS
  thresh=0.40: TT= 46, TF= 87, acc=0.3459, sparsity=0.1218 PASS
  thresh=0.45: TT= 35, TF= 98, acc=0.2632, sparsity=0.0850 PASS
  thresh=0.50: TT= 24, TF=109, acc=0.1805, sparsity=0.0524 PASS

Best threshold: 0.32 (chosen to also pass training sparsity)
--- Stacked: Cluster 2 — 5-Fold CV (thresh=0.32) ---
TT (bankrupt predicted bankrupt):     67
TF (bankrupt predicted non-bankrupt): 66
TT + TF = 133 (total bankrupt)
Custom Accuracy (TT/(TT+TF)): 0.5038
Sparsity: 0.1

(np.float64(0.5037593984962406), np.float64(0.19688385269121814))

## Train Final Model & Save

Fit the scaler on the full cluster, fit the stacking classifier, then predict on the
full cluster_2.csv (all 706 rows). These are the Table 3 numbers.

In [8]:
# Fit final scaler + stacking model on all cluster 2 data
scaler_final = StandardScaler()
X2_scaled = scaler_final.fit_transform(X2_sel)

stacker_final = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=200, max_depth=6, min_samples_leaf=5,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ('gb', GradientBoostingClassifier(
            n_estimators=150, max_depth=3, learning_rate=0.05,
            min_samples_leaf=10, random_state=RANDOM_STATE)),
        ('knn', KNeighborsClassifier(
            n_neighbors=20, weights='distance', n_jobs=-1)),
    ],
    final_estimator=LogisticRegression(
        C=2.0, max_iter=1000, random_state=RANDOM_STATE),
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    stack_method='predict_proba',
    n_jobs=-1
)
stacker_final.fit(X2_scaled, y2)

# Predict on original cluster 2 data — Table 3 numbers
y2_train_proba = stacker_final.predict_proba(X2_scaled)[:, 1]
y2_train_pred = (y2_train_proba >= BEST_THRESH).astype(int)
evaluate_model(y2, y2_train_pred, f'Stacked: Cluster 2 — Training on full cluster (thresh={BEST_THRESH:.2f})')

# Also print individual base-model training CMs from the fitted stack
print('\n--- Individual Base Models (trained, full data, thresh=0.5) ---')
for name, est in stacker_final.named_estimators_.items():
    base_preds = est.predict(X2_scaled)
    evaluate_model(y2, base_preds, f'Base(trained): {name}')

--- Stacked: Cluster 2 — Training on full cluster (thresh=0.32) ---
TT (bankrupt predicted bankrupt):     132
TF (bankrupt predicted non-bankrupt): 1
TT + TF = 133 (total bankrupt)
Custom Accuracy (TT/(TT+TF)): 0.9925
Sparsity: 0.1941 (PASS)
Confusion Matrix:
[[568   5]
 [  1 132]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       573
           1       0.96      0.99      0.98       133

    accuracy                           0.99       706
   macro avg       0.98      0.99      0.99       706
weighted avg       0.99      0.99      0.99       706


--- Individual Base Models (trained, full data, thresh=0.5) ---
--- Base(trained): rf ---
TT (bankrupt predicted bankrupt):     115
TF (bankrupt predicted non-bankrupt): 18
TT + TF = 133 (total bankrupt)
Custom Accuracy (TT/(TT+TF)): 0.8647
Sparsity: 0.2890 (FAIL)
Confusion Matrix:
[[484  89]
 [ 18 115]]
Classification Report:
              precision    recall  f1

In [9]:
# Save cluster 2 pipeline (scaler + stacking model + metadata)
model_info = {
    'model': stacker_final,
    'scaler': scaler_final,
    'features': top_features,
    'n_features': N_FEATURES,
    'threshold': BEST_THRESH,
    'cluster_id': 2,
    'model_type': 'stacking',
    'base_models': 'RF + GB + KNN(k=20, distance)',
    'meta_model': 'LogisticRegression(C=2.0)',
    'member': 'Sanjavan Ghodasara'
}
joblib.dump(model_info, os.path.join(SUBGROUP_MODEL_DIR, 'cluster_2_model.joblib'))
print(f'Cluster 2 model saved. Features: {N_FEATURES}, Threshold: {BEST_THRESH:.2f}')
print(f'Saved keys: {sorted(model_info.keys())}')

Cluster 2 model saved. Features: 6, Threshold: 0.32
Saved keys: ['base_models', 'cluster_id', 'features', 'member', 'meta_model', 'model', 'model_type', 'n_features', 'scaler', 'threshold']


## Summary Cluster 2

| Metric | CV (5-fold) | Training |
|---|---|---|
| TT (bankrupt predicted bankrupt) | **67** | **132** |
| TF (bankrupt predicted non-bankrupt) | **66** | **1** |
| TT + TF | 133 | 133 |
| Custom Accuracy (TT/(TT+TF)) | **50.38%** | **99.25%** |
| Sparsity (% predicted bankrupt) | 19.69% **PASS** | 19.41% **PASS** |
| Features (N) | 6 | 6 |
| Threshold | 0.32 | 0.32 |
| Feature Score ((50-N)/50) | 0.88 | 0.88 |

**Estimated Rank Score:** 0.3(0.9925) + 0.4(0.5038) + 0.3(0.88) = **0.7632**

**Improvement over baseline** (RF+GB+SVM, LR C=0.1): **+0.1263** (from 0.6369 → 0.7632).

**Top 6 Features:**
1. Total debt/Total net worth
2. Equity to Liability
3. Debt ratio %
4. Net profit before tax/Paid-in capital
5. Net Income to Stockholder's Equity
6. Working Capital to Total Assets

**Final Model:** Stacking (RF + GB + KNN[k=20, distance-weighted] → LR C=2.0 meta), no SMOTE, 5-fold CV, per-fold StandardScaler.

### Saved Artifacts
- `models/subgroup_models/cluster_2_model.joblib` — contains `model` (fitted stack), `scaler` (StandardScaler), `features` (list), `threshold` (0.32), and metadata.

### Why KNN made the difference for cluster 2

Cluster 2 is the high-risk cluster (18.84% bankruptcy rate). Distance-weighted KNN (k=20)
captures the local density of bankruptcy cases in feature space — each query point's
prediction is pulled toward the class of its nearest neighbors, weighted by inverse distance.
The trees (RF, GB) model global/axis-aligned splits; KNN models a different, complementary signal,
which gives the stack's logistic meta-learner a richer feature basis to combine.